In [0]:
from pyspark.sql.functions import col

reviews = spark.read.table("airbnb_joao_pessoa.bronze.reviews")
reviews.printSchema()

In [0]:
print(f"Quantidade de registros: {reviews.count()}")
print(f"Quantidade de colunas: {len(reviews.columns)}")

In [0]:
from pyspark.sql.functions import sum, when

null_summary = reviews.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in reviews.columns])
display(null_summary)

In [0]:
duplicated_reviews = (reviews.groupBy("listing_id", "date").count().filter(col("count") > 1))

display(duplicated_reviews)

In [0]:
from pyspark.sql.functions import min, max

display(reviews.select(min("date").alias("min_date"), max("date").alias("max_date")))

In [0]:
display(reviews.select("num_reviews").summary())

In [0]:
display(reviews.filter(col("num_reviews") < 0).select("listing_id", "date", "num_reviews"))

In [0]:
# ANalisar reviewers
from pyspark.sql.functions import size

reviews_check = reviews.withColumn("reviewers_count", size("reviewers"))

display(reviews_check.select("listing_id", "date", "num_reviews", "reviewers_count"))

In [0]:
# Comparar num_reviews com a quantidade de reviewers
display(reviews_check
    .filter(col("num_reviews").isNotNull() & col("reviewers").isNotNull() & (col("num_reviews") != col("reviewers_count")))
    .select("listing_id", "date", "num_reviews", "reviewers_count", "reviewers"))

In [0]:
# Verificar array vazios
display(reviews_check
    .filter(col("reviewers").isNotNull() & (size("reviewers") == 0))
    .select("listing_id", "date", "num_reviews", "reviewers"))